In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

In [2]:
# Load all datasets
brokers = pd.read_csv("synthetic_brokers_v80.csv")
leads = pd.read_csv("synthetic_leads_v80.csv")
assignments = pd.read_csv("synthetic_assignments_v80.csv")
counterfactual = pd.read_csv("synthetic_counterfactual_v80.csv")
historical = pd.read_csv("synthetic_historical_v80.csv")

# Display basic information
datasets = {
    "brokers": brokers,
    "leads": leads,
    "assignments": assignments,
    "counterfactual": counterfactual,
    "historical": historical
}



/tmp/ipykernel_27789/3568090743.py:6: DtypeWarning: Columns (33,35) have mixed types. Specify dtype option on import or set low_memory=False.
  historical = pd.read_csv("synthetic_historical_v80.csv")


In [3]:
reentry_pattern = r'_R\d+$'
reentry_lead_ids = assignments[
    assignments['lead_id'].str.contains(reentry_pattern, na=False)
]['lead_id'].unique()
print(f"Found {len(reentry_lead_ids)} re-entry lead IDs in assignments")

Found 850 re-entry lead IDs in assignments


In [4]:
assignments['original_lead_id_from_assign'] = assignments.apply(
    lambda row: row.get('original_lead_id', row['lead_id'].split('_R')[0] if '_R' in str(row['lead_id']) else None),
    axis=1
)

# Create re-entry leads dataframe by merging with original lead data
reentry_leads = []
for reentry_id in reentry_lead_ids:
    # Find the original lead ID
    original_id = reentry_id.split('_R')[0]
    
    # Get original lead data
    original_lead = leads[leads['lead_id'] == original_id]
    if len(original_lead) > 0:
        reentry_lead = original_lead.iloc[0].to_dict()
        reentry_lead['lead_id'] = reentry_id
        reentry_lead['original_lead_id'] = original_id
        
        # Add re-entry specific fields
        reentry_lead['is_reentry'] = True
        reentry_lead['reentry_number'] = int(reentry_id.split('_R')[1]) if '_R' in reentry_id else 0
        
        reentry_leads.append(reentry_lead)

# Combine original and re-entry leads
reentry_leads_df = pd.DataFrame(reentry_leads)
leads_full = pd.concat([leads, reentry_leads_df], ignore_index=True)
print(f"Original leads: {len(leads):,}")
print(f"Re-entry leads added: {len(reentry_leads):,}")
print(f"Total leads: {len(leads_full):,}")


Original leads: 10,624
Re-entry leads added: 850
Total leads: 11,474


In [5]:
broker_num_pattern = r'BR-0(\d+)'
brokers['broker_num'] = brokers['broker_id'].str.extract(broker_num_pattern).astype(float)
max_original_broker = brokers['broker_num'].max()
print(f"Original brokers: {len(brokers)} (max ID: BR-{int(max_original_broker):04d})")

# Find replacement broker IDs in assignments
assignment_brokers = assignments['broker_id'].unique()
replacement_broker_ids = [
    bid for bid in assignment_brokers 
    if int(bid.split('-')[1]) > max_original_broker
]

print(f"Found {len(replacement_broker_ids)} replacement broker IDs in assignments")

# Create replacement broker profiles based on new broker characteristics
new_broker_profiles = brokers[brokers['is_new_broker'] == True].copy()

replacement_brokers = []
for broker_id in replacement_broker_ids:
    # Sample from existing new broker profiles
    base_profile = new_broker_profiles.sample(1).iloc[0].to_dict()
    
    # Modify to be realistic for replacement brokers
    base_profile['broker_id'] = broker_id
    base_profile['is_new_broker'] = True
    base_profile['is_replacement'] = True
    base_profile['years_experience'] = 0
    base_profile['skill_level'] = np.clip(base_profile['skill_level'] + np.random.normal(0, 0.05), 0.3, 0.8)
    base_profile['conversion_rate'] = np.clip(base_profile['conversion_rate'] + np.random.normal(0, 0.02), 0.05, 0.55)
    
    replacement_brokers.append(base_profile)

replacement_brokers_df = pd.DataFrame(replacement_brokers)
brokers_full = pd.concat([brokers, replacement_brokers_df], ignore_index=True)
print(f"Original brokers: {len(brokers):,}")
print(f"Replacement brokers added: {len(replacement_brokers):,}")
print(f"Total brokers: {len(brokers_full):,}")

Original brokers: 300 (max ID: BR-0300)
Found 6 replacement broker IDs in assignments
Original brokers: 300
Replacement brokers added: 6
Total brokers: 306


In [6]:
brokers_full['utilization'] = brokers_full['current_caseload'] / brokers_full['capacity']
print(f"Broker utilization - mean: {brokers_full['utilization'].mean():.2f}")
print(f"Broker utilization - 95th percentile: {brokers_full['utilization'].quantile(0.95):.2f}")
print(f"Brokers exceeding capacity: {(brokers_full['utilization'] > 1).sum():,} / {len(brokers_full):,}")


Broker utilization - mean: 0.46
Broker utilization - 95th percentile: 0.93
Brokers exceeding capacity: 0 / 306


In [7]:
# Merge with leads_full and brokers_full
merged = assignments.merge(
    leads_full, 
    on='lead_id', 
    how='left',
    suffixes=('', '_lead')
)

merged = merged.merge(
    brokers_full,
    on='broker_id',
    how='left',
    suffixes=('', '_broker')
)

# Add conversion from historical
if 'converted' in historical.columns:
    merged = merged.merge(
        historical[['lead_id', 'broker_id', 'converted']],
        on=['lead_id', 'broker_id'],
        how='left'
    )

# Add flags for data completeness
merged['has_original_lead'] = merged['lead_id'].isin(leads['lead_id'])
merged['has_original_broker'] = merged['broker_id'].isin(brokers['broker_id'])
merged['is_reentry_lead'] = merged['lead_id'].str.contains(reentry_pattern, na=False)
merged['is_replacement_broker'] = merged['broker_id'].isin(replacement_broker_ids)

print(f"Final merged dataset shape: {merged.shape}")
print(f"Records with original lead data: {merged['has_original_lead'].sum():,}")
print(f"Records with original broker data: {merged['has_original_broker'].sum():,}")

Final merged dataset shape: (54136, 72)
Records with original lead data: 50,143
Records with original broker data: 53,558


In [8]:
missing_before = merged.isnull().sum()
missing_pct_before = (missing_before / len(merged)) * 100
print("\nMissing data before handling:")
print(missing_pct_before[missing_pct_before > 0].sort_values(ascending=False).round(2))



Missing data before handling:
is_replacement              98.93
conversion_delay_days       95.61
reentry_number              92.62
original_lead_id_lead       92.62
is_reentry                  92.62
response_time_hours         68.83
responded                   62.40
insurance_type               2.15
language                     2.14
tenure_years                 1.30
digital_engagement_score     0.59
dtype: float64


In [9]:
lead_cols_with_missing = ['insurance_type', 'language', 'tenure_years', 'digital_engagement_score']

for col in lead_cols_with_missing:
    if col in merged.columns:
        merged[f'{col}_missing'] = merged[col].isna().astype(int)
        print(f"  Created flag: {col}_missing (missing rate: {merged[col].isna().mean()*100:.2f}%)")

  Created flag: insurance_type_missing (missing rate: 2.15%)
  Created flag: language_missing (missing rate: 2.14%)
  Created flag: tenure_years_missing (missing rate: 1.30%)
  Created flag: digital_engagement_score_missing (missing rate: 0.59%)


In [10]:
for col in ['insurance_type', 'language']:
    if col in merged.columns:
        merged[col] = merged[col].fillna('UNKNOWN')
        print(f"  Imputed {col} with 'UNKNOWN'")

# For numeric lead data, impute with median (but keep flags)
for col in ['tenure_years', 'digital_engagement_score']:
    if col in merged.columns:
        median_val = merged[col].median()
        merged[col] = merged[col].fillna(median_val)
        print(f"  Imputed {col} with median: {median_val:.2f}")

  Imputed insurance_type with 'UNKNOWN'
  Imputed language with 'UNKNOWN'
  Imputed tenure_years with median: 1.00
  Imputed digital_engagement_score with median: 34.20


In [11]:
assignment_design_cols = ['conversion_delay_days', 'response_time_hours', 'responded']

for col in assignment_design_cols:
    if col in merged.columns:
        missing_count = merged[col].isna().sum()
        missing_pct = (missing_count / len(merged)) * 100
        print(f"  {col}: {missing_count:,} missing ({missing_pct:.2f}%) - kept as NaN")


  conversion_delay_days: 51,759 missing (95.61%) - kept as NaN
  response_time_hours: 37,263 missing (68.83%) - kept as NaN
  responded: 33,782 missing (62.40%) - kept as NaN


In [13]:
numeric_cols = merged.select_dtypes(include=[np.number]).columns.tolist()

# Exclude columns we've already handled
exclude_cols = lead_cols_with_missing + assignment_design_cols + [f'{c}_missing' for c in lead_cols_with_missing]
other_missing_numeric = [
    col for col in numeric_cols 
    if merged[col].isna().any() 
    and col not in exclude_cols
]

print(f"Found {len(other_missing_numeric)} numeric columns with missing data to handle")

for col in other_missing_numeric:
    missing_count = merged[col].isna().sum()
    missing_pct = (missing_count / len(merged)) * 100
    
    if missing_pct < 1:
        # Drop rows with missing data if very few
        merged = merged.dropna(subset=[col])
        print(f"  {col}: dropped {missing_count:,} rows ({missing_pct:.2f}%)")
    else:
        # Impute with median for larger missingness
        median_val = merged[col].median()
        merged[col] = merged[col].fillna(median_val)
        print(f"  {col}: imputed with median {median_val:.2f} ({missing_pct:.2f}%)")


Found 1 numeric columns with missing data to handle
  reentry_number: imputed with median 420.00 (92.62%)
